# Phase 4 — Document Extraction

**Goal:** turn the 12 raw files in `data/raw/` into clean text, with tables
preserved as markdown and every page tagged with its department, version,
and OCR status.

**Why this phase matters:** everything downstream — chunking, retrieval,
access control, citations — depends on what comes out of this notebook.
If a table gets flattened here, the LLM will report a wrong number to an
employee. If metadata isn't attached here, access control can't be
enforced later.

**Structure of this notebook:**
1. Setup — imports, config, load the manifest
2. `table_to_markdown` — turn a raw table into an LLM-readable format
3. `extract_page` — reconstruct one PDF page in reading order (text + tables)
4. `ocr_page` — fallback for scanned pages with no text layer
5. `extract_pdf` — combine 3 + 4 across a whole PDF, with per-page OCR detection
6. `extract_docx` — same idea for Word files, which have no fixed pages
7. `extract_document` — unified entry point: dispatch on file type, attach manifest metadata
8. Run across the full corpus and validate the result


## 1. Setup

Imports, one config constant, and the corpus manifest (the CSV that maps
each file to its department, version, and current/superseded status —
see `data/corpus_manifest.csv`).


In [1]:
from pathlib import Path

import pandas as pd
import pdfplumber
from pypdf import PdfReader

from pdf2image import convert_from_path
import pytesseract

from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph

In [4]:
# Points to trim from the bottom of every page, to strip the repeated
# footer ("Nilachala Textiles Pvt. Ltd. - Internal document...").
# 60pt was measured against this project's own PDF margins.
FOOTER_ZONE = 60

RAW_DIR = Path("/home/lenovo/projects/nilachala-policy-assistant/data/raw")
MANIFEST_PATH = Path("/home/lenovo/projects/nilachala-policy-assistant/data/corpus_manifest.csv")

In [5]:
manifest = pd.read_csv(MANIFEST_PATH).set_index("doc_id")
print(f"Documents in manifest: {len(manifest)}")
manifest[["title", "department", "format", "is_scanned", "is_current"]]

Documents in manifest: 12


,title,department,format,is_scanned,is_current
doc_id,,,,,
D01,Casual and Earned Leave Policy,ALL,pdf,False,True
D02,Leave Policy,ALL,pdf,False,False
D03,Employee Handbook,ALL,pdf,False,True
D04,Factory Floor Safety Manual,ALL,pdf,True,True
D05,Fire and Evacuation Procedure,ALL,pdf,True,True
D06,Machine Operating Guidelines,Production,pdf,True,True
D07,Laptop and IT Asset Policy,ALL,pdf,False,True
D08,IT Support and Escalation Matrix,IT,docx,False,True
D09,Travel and Expense Reimbursement Policy,ALL,pdf,False,True


## 2. `table_to_markdown` — preserve table structure

`pypdf` reads a table as a flat stream of tokens with no row boundaries —
the LLM can't tell which number belongs to which row. `pdfplumber` can
detect the actual grid, but hands it back as a list of lists. This
function turns that list of lists into a markdown table, which is the
format LLMs read most reliably.

Two edge cases handled: `None` for empty cells (pdfplumber's default,
which would otherwise print the literal word "None"), and embedded
newlines from wrapped cell text (which would otherwise break a row across
two lines and corrupt the table).


In [6]:
def table_to_markdown(table):
    """Convert a pdfplumber/docx table (list of lists) into a markdown string."""
    if not table:
        return ""

    def clean(cell):
        if cell is None:
            return ""
        return str(cell).replace("\n", " ").replace("|", "\\|").strip()

    lines = []
    for i, row in enumerate(table):
        lines.append("| " + " | ".join(clean(c) for c in row) + " |")
        if i == 0:
            lines.append("| " + " | ".join("---" for _ in row) + " |")

    return "\n".join(lines)

In [9]:
# Sanity check against a real table from the corpus
with pdfplumber.open(RAW_DIR / "casual_earned_leave_policy_v3.pdf") as pdf:
    raw_table = pdf.pages[0].extract_tables()[0]

print(raw_table)
print("Table to markdown:")
print(table_to_markdown(raw_table))

[['Leave type', 'Days per year', 'Carry forward', 'Encashable'], ['Casual leave', '12', 'Not permitted', 'No'], ['Earned leave', '18', 'Up to 30 days', 'Yes, on separation'], ['Sick leave', '10', 'Not permitted', 'No'], ['Maternity leave', '182', 'Not applicable', 'No'], ['Paternity leave', '15', 'Not applicable', 'No'], ['Bereavement leave', '5', 'Not applicable', 'No']]
Table to markdown:
| Leave type | Days per year | Carry forward | Encashable |
| --- | --- | --- | --- |
| Casual leave | 12 | Not permitted | No |
| Earned leave | 18 | Up to 30 days | Yes, on separation |
| Sick leave | 10 | Not permitted | No |
| Maternity leave | 182 | Not applicable | No |
| Paternity leave | 15 | Not applicable | No |
| Bereavement leave | 5 | Not applicable | No |


## 3. `extract_page` — reconstruct one page in reading order

`pdfplumber` gives page text and page tables as two separate things, with
no indication of where a table sits relative to the surrounding text. If
you just append tables to the end, they land disconnected from the
heading that explains them — a chunk of "Rs. 1,400 / Rs. 1,100" with no
indication that it's a *daily allowance* table.

The fix: every table object has a bounding box (`.bbox`) giving its
vertical position on the page. Walk the page top to bottom, and whenever
the cursor reaches a table's position, emit markdown instead of raw text
for that band. Finally, stop short of the bottom `FOOTER_ZONE` points so
the repeated footer line never enters a chunk.


In [12]:
def extract_page(page):
    """Reconstruct a pdfplumber page as text, with tables inlined as
    markdown at their correct position, and the footer band excluded."""
    tables = page.find_tables()
    usable_bottom = page.height - FOOTER_ZONE

    parts = []
    cursor = 0
    for t in tables:
        top, bottom = t.bbox[1], t.bbox[3]
        strip = page.crop((0, cursor, page.width, top))
        parts.append(strip.extract_text() or "")
        parts.append(table_to_markdown(t.extract()))
        cursor = bottom

    # remaining text below the last table, stopping before the footer
    parts.append(page.crop((0, cursor, page.width, usable_bottom)).extract_text() or "")

    return "\n\n".join(p.strip() for p in parts if p.strip())

In [11]:
# Test on page 1 of the travel policy - it has two tables on one page,
# which is the layout most likely to expose a positioning bug.
with pdfplumber.open(RAW_DIR / "travel_expense_reimbursement.pdf") as pdf:
    print(extract_page(pdf.pages[0]))

Travel and Expense Reimbursement Policy
Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D09 | Version 2.0 | Owner: ALL department
1. Scope and Principle
This policy covers expenses incurred by employees while travelling on company business.
Employees are expected to exercise the same care in incurring expenses on company business as they
would in managing their own affairs. Expenses must be reasonable, necessary and supported by
evidence.
2. Prior Approval
All business travel requires prior written approval from the department head using Form FIN-12.
International travel additionally requires the approval of the Managing Director.
3. Travel Entitlement by Grade
Table 3.1 - Mode of travel entitlement

| Grade | Rail | Air | Local transport |
| --- | --- | --- | --- |
| Grade 1 - 2 | Sleeper class | Not permitted | Bus or shared auto |
| Grade 3 - 4 | AC 3 tier | Not permitted | Auto rickshaw |
| Grade 5 - 6 | AC 2 tier | Economy, over 500 km | Taxi |
| Grade 7 and above | AC 1 t

## 4. `ocr_page` — fallback for scanned documents

Three documents in the corpus (`factory_floor_safety_manual.pdf`,
`fire_evacuation_procedure.pdf`, `machine_operating_guidelines.pdf`) are
image-only PDFs with no extractable text layer at all — the digital
equivalent of a photocopy. `extract_page` will return an empty string for
these, so they need a different path: rasterise the page, then run OCR.

`dpi=300` matters — Tesseract was trained on roughly 300 DPI scans;
150 DPI measurably hurts accuracy. The crop uses the same `FOOTER_ZONE`
concept as `extract_page`, but converted from PDF points to pixels at
this DPI, since the two use different units.


In [13]:
def ocr_page(pdf_path, page_number, dpi=300):
    """OCR a single page of a scanned PDF (1-indexed page_number)."""
    images = convert_from_path(
        str(pdf_path), dpi=dpi,
        first_page=page_number, last_page=page_number,
    )
    img = images[0]

    footer_px = int(FOOTER_ZONE * dpi / 72)   # 72 points per inch
    img = img.crop((0, 0, img.width, img.height - footer_px))

    return pytesseract.image_to_string(img)

In [14]:
# Test on a genuinely scanned document
text = ocr_page(RAW_DIR / "fire_evacuation_procedure.pdf", page_number=1)
print(text[:600])

Fire and Evacuation Procedure

Nilachala Textiles Pvt. Ltd., Bhubaneswar

Document ref: DO5 Version 1.0 Owner: ALL department

1. On Discovering a Fire

Raise the alarm immediately by operating the nearest manual call point. Call points are located at each
stairwell and at both ends of every production hall.

Inform the security office on extension 100. State your name, your location and the nature of the fire.

Attempt to extinguish the fire only if it is small, only if you have been trained, and only if you can do so
without placing yourself at risk. Never attempt to fight a fire that is bet


**Note on accuracy:** OCR is probabilistic, not exact. Read the output
above closely and you'll likely spot at least one character error (in the
original run of this project, the document reference `D05` came back as
`DO5` — the letter O for the digit 0). This is exactly why the pipeline
tracks a `used_ocr` flag per page: any answer sourced from an OCR'd page —
especially a safety document — should carry a "verify against the printed
original" notice rather than being trusted at face value.


## 5. `extract_pdf` — the full PDF path

Combines the two paths above. For each page: try `extract_page` first;
if it returns fewer than ~50 characters, treat that as "no usable text
layer" and fall back to `ocr_page`. This check runs **per page**, not per
document — a real-world corpus can have a digital PDF with one scanned
insert page, and this handles that correctly.

Returns a list of `(page_number, text, used_ocr)` tuples — page number
kept explicit because it's what powers citations later.


In [15]:
def extract_pdf(path):
    """Extract every page of a PDF, falling back to OCR per-page as needed."""
    with pdfplumber.open(path) as pdf:
        pages = []
        for i, page in enumerate(pdf.pages, start=1):
            text = extract_page(page)
            used_ocr = False
            if len(text) < 50:
                text = ocr_page(path, i)
                used_ocr = True
            pages.append((i, text, used_ocr))
    return pages

In [16]:
# Run across every PDF in the corpus and confirm the OCR fallback fires
# only for the three genuinely scanned documents.
for pdf_file in sorted(RAW_DIR.glob("*.pdf")):
    pages = extract_pdf(pdf_file)
    total_chars = sum(len(t) for _, t, _ in pages)
    ocr_count = sum(1 for _, _, used in pages if used)
    print(f"{pdf_file.name:42s} {len(pages):2d} pages  {total_chars:6d} chars  {ocr_count} via OCR")

casual_earned_leave_policy_v3.pdf           2 pages    3324 chars  0 via OCR
employee_handbook.pdf                      11 pages   20475 chars  0 via OCR
factory_floor_safety_manual.pdf             2 pages    2978 chars  2 via OCR
fire_evacuation_procedure.pdf               2 pages    2465 chars  2 via OCR
laptop_it_asset_policy.pdf                  2 pages    2293 chars  0 via OCR
leave_policy_2019.pdf                       1 pages    1157 chars  0 via OCR
machine_operating_guidelines.pdf            1 pages    1772 chars  1 via OCR
provident_fund_gratuity_guidelines.pdf      2 pages    2572 chars  0 via OCR
salary_grade_structure.pdf                  2 pages    2145 chars  0 via OCR
travel_expense_reimbursement.pdf            2 pages    2322 chars  0 via OCR


## 6. `extract_docx` — Word documents

`python-docx` exposes paragraphs and tables as two *separate* lists, so
naively reading `doc.paragraphs` then `doc.tables` loses the interleaving
— same positioning problem as the PDF path, different cause.

The fix: walk `doc.element.body.iterchildren()`, which iterates the
underlying XML in true document order, and branch on the tag (`}p` for
paragraph, `}tbl` for table).

**Important limitation:** DOCX has no fixed pagination — page breaks are
computed by whatever renders the file, not stored in it. So this always
returns a single "page" (page number `1`) containing the whole document.
Citations for these two files can name the document and section, but not
a meaningful page number.


In [17]:
def extract_docx(path):
    """Extract a DOCX in document order, tables inlined as markdown.
    Returns a single (page=1, text, used_ocr=False) tuple - DOCX has no
    fixed pagination."""
    doc = Document(str(path))
    parts = []

    for child in doc.element.body.iterchildren():
        if child.tag.endswith("}p"):
            text = Paragraph(child, doc).text.strip()
            if text:
                parts.append(text)
        elif child.tag.endswith("}tbl"):
            table = Table(child, doc)
            rows = [[cell.text for cell in row.cells] for row in table.rows]
            parts.append(table_to_markdown(rows))

    return [(1, "\n\n".join(parts), False)]

In [18]:
# Test on the DOCX with two tables - the SLA priority matrix and the
# escalation levels - to confirm both interleave correctly with the text.
pages = extract_docx(RAW_DIR / "it_support_escalation_matrix.docx")
print(pages[0][1])

IT Support and Escalation Matrix

Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D08  |  Version 1.2  |  Owner: IT department

1. Purpose

This document defines the internal service levels of the IT department and the escalation path for unresolved incidents.

This document is for the internal use of the IT department. It is not circulated to other departments.

2. Incident Priority Definitions

Table 2.1 - Priority definitions and response targets

| Priority | Definition | Response time | Resolution target |
| --- | --- | --- | --- |
| P1 | Production line stopped or full network outage | 15 minutes | 4 hours |
| P2 | Business function degraded, workaround exists | 1 hour | 1 working day |
| P3 | Single user unable to work | 4 hours | 2 working days |
| P4 | Request or minor issue, no work impact | 1 working day | 5 working days |

Priority is assigned by the IT helpdesk on logging and may be revised by the IT Manager. The requester may request a review of the assigned prior

## 7. `extract_document` — unified entry point

`extract_pdf` and `extract_docx` both return the same shape:
`(page_number, text, used_ocr)`. This function is the join point between
that raw content and the manifest metadata — it dispatches on file
extension, then rebuilds each page tuple into a dict carrying everything
downstream code needs:

- `department` → will drive access-control filtering at retrieval time
- `is_current` → excludes superseded documents (like the 2019 leave
  policy) from ever being retrieved
- `used_ocr` → triggers the "verify against printed original" notice
- `page` → powers citations

**Input:** a file path, and one row of the manifest (as a pandas Series,
via `manifest.loc[doc_id]`).
**Output:** a list of dicts, one per page.


In [21]:
def extract_document(file_path, row):
    """Extract a document and attach its manifest metadata to every page.

    file_path : path to the file in data/raw/
    row       : one row of the manifest, e.g. manifest.loc["D01"]
    """
    ext = Path(file_path).suffix.lower()

    if ext == ".pdf":
        raw_pages = extract_pdf(file_path)
    elif ext == ".docx":
        raw_pages = extract_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    pages = []
    for page_num, text, used_ocr in raw_pages:
        pages.append({
            "text": text,
            "page": page_num,
            "used_ocr": used_ocr,
            "doc_id": row.name,
            "source_file": row["filename"],
            "title": row["title"],
            "department": row["department"],
            "version": str(row["version"]),
            "is_current": str(row["is_current"]).strip().upper() == "TRUE",
        })
    return pages

In [22]:
# Test on a single document first
row = manifest.loc["D01"]
pages = extract_document(RAW_DIR / row["filename"], row)

print(f"Pages returned: {len(pages)}")
print(f"is_current type: {type(pages[0]['is_current'])}  value: {pages[0]['is_current']}")
print()
print(pages[0]["text"][:300])

Pages returned: 2
is_current type: <class 'bool'>  value: True

Casual and Earned Leave Policy
Nilachala Textiles Pvt. Ltd., Bhubaneswar
Document ref: D01 | Version 3.0 | Owner: ALL department
1. Purpose and Scope
This policy defines the leave entitlement available to all confirmed employees of Nilachala Textiles Pvt.
Ltd. and the procedure for applying for and 


## 8. Run across the full corpus

The end-to-end check for this phase. Every document in the manifest gets
extracted; the summary at the end confirms four things at once:

- every document produced at least one page
- OCR fired for exactly the three scanned documents (5 pages total)
- all four departments are represented, so access-control filtering has
  something to filter on
- **D02 — and only D02 — is flagged as superseded.** This is the
  deliberate version conflict planted in the corpus (D01 says 12 casual /
  18 earned leave days; D02 says 10 / 15). If this list ever shows more
  than `['D02']`, or is empty, something upstream is wrong.


In [23]:
all_pages = []

for doc_id, row in manifest.iterrows():
    pages = extract_document(RAW_DIR / row["filename"], row)
    all_pages.extend(pages)
    ocr_count = sum(1 for p in pages if p["used_ocr"])
    print(f"{doc_id}  {len(pages):2d} pages  {ocr_count} via OCR   {row['title'][:42]}")

print(f"\nTotal pages : {len(all_pages)}")
print(f"Total chars : {sum(len(p['text']) for p in all_pages):,}")
print(f"Departments : {sorted(set(p['department'] for p in all_pages))}")
print(f"Superseded  : {[p['doc_id'] for p in all_pages if not p['is_current']]}")

D01   2 pages  0 via OCR   Casual and Earned Leave Policy
D02   1 pages  0 via OCR   Leave Policy
D03  11 pages  0 via OCR   Employee Handbook
D04   2 pages  2 via OCR   Factory Floor Safety Manual
D05   2 pages  2 via OCR   Fire and Evacuation Procedure
D06   1 pages  1 via OCR   Machine Operating Guidelines
D07   2 pages  0 via OCR   Laptop and IT Asset Policy
D08   1 pages  0 via OCR   IT Support and Escalation Matrix
D09   2 pages  0 via OCR   Travel and Expense Reimbursement Policy
D10   2 pages  0 via OCR   Salary Grade and Allowance Structure
D11   2 pages  0 via OCR   Provident Fund and Gratuity Guidelines
D12   1 pages  0 via OCR   Code of Conduct and Disciplinary Procedure

Total pages : 29
Total chars : 47,071
Departments : ['ALL', 'Finance', 'IT', 'Production']
Superseded  : ['D02']


---

**Phase 4 complete.** Every document in the corpus — digital PDF, scanned
PDF, and DOCX — now produces clean, position-correct text with tables
preserved as markdown and full metadata attached to every page.

**Next (Phase 5, new notebook `02_chunking.ipynb`):** these page-level
blocks are still too large to embed well, and some sections span page
boundaries. Chunking needs to split on structure (numbered headings)
while never splitting a table, and each resulting chunk needs to inherit
this metadata plus a `chunk_id`.
